# Hands-on DDPM: Colab Training Driver

This notebook trains the from-scratch DDPM on a Colab GPU by calling the **same `ddpm` CLI** used locally. It contains no model, loss or training-loop code (Constitution *Prohibited Patterns*; ADR 0001 D3).

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Add a Colab Secret named **`GH_TOKEN`** (key icon in the left sidebar): a GitHub token with read access to the private repository `Abd-Elfattah5/Hands-on-DDPM`, with notebook access enabled. *Alternative*: upload `Hands-on-DDPM.zip` to `MyDrive/hands-on-ddpm/`.
3. **Runtime → Run all**.
4. If the session disconnects, reconnect and **Run all** again: training resumes from `latest.pt` on Google Drive (at most one epoch is lost).

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU: select Runtime → Change runtime type → T4 GPU, then rerun."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | {props.total_memory / 1024**3:.1f} GB | torch {torch.__version__}")

GPU: Tesla T4 | 14.6 GB | torch 2.11.0+cu130


In [ ]:
# 2. Mount Google Drive (checkpoints and metrics live here so they survive disconnects)
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/hands-on-ddpm"
RUN_DIR = f"{DRIVE_ROOT}/runs/cifar10_colab"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
REPO_DIR = "/content/Hands-on-DDPM"
BRANCH = "main"  # set to a feature branch name to train unmerged code (the reported run used "001-create-ddpm")
import os
os.makedirs(RUN_DIR, exist_ok=True)
print("Run directory:", RUN_DIR)

Mounted at /content/drive
Run directory: /content/drive/MyDrive/hands-on-ddpm/runs/cifar10_colab


In [ ]:
# 3. Get the code: clone with the GH_TOKEN secret, or unzip a copy from Drive
import os, subprocess, zipfile
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
    except Exception as exc:
        print("GH_TOKEN secret not available:", type(exc).__name__)
    if token:
        url = f"https://{token}@github.com/Abd-Elfattah5/Hands-on-DDPM.git"
        result = subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError("git clone failed: " + result.stderr.replace(token, "***"))
        # Remove the token from the stored remote URL (git saves it in .git/config otherwise).
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                        "https://github.com/Abd-Elfattah5/Hands-on-DDPM.git"], check=True)
        print(f"Cloned branch {BRANCH}")
    else:
        archive = f"{DRIVE_ROOT}/Hands-on-DDPM.zip"
        assert os.path.exists(archive), f"Add the GH_TOKEN secret or upload {archive}"
        with zipfile.ZipFile(archive) as zf:
            zf.extractall("/content")
        print("Extracted", archive)
    token = None  # drop the secret from the notebook namespace
%cd {REPO_DIR}
!git log --oneline -1 || true

Cloned branch 001-create-ddpm
/content/Hands-on-DDPM
bc7ca0b (HEAD -> 001-create-ddpm, origin/001-create-ddpm) fix: resolve branch review findings R1-R6


In [ ]:
# 4. Install the package (Colab's preinstalled PyTorch is used)
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hands-on-ddpm (pyproject.toml) ... done


In [ ]:
# 5. Pre-training gate (memory check skipped: the 3 GB budget applies to the local GPU)
!ddpm verify --config configs/cifar10_colab.yaml --skip-memory

Verifying configs/cifar10_colab.yaml
  ✓ Configuration validates against the schema
  ✓ Schedule: β ∈ (0,1), ᾱ_t strictly decreasing, ᾱ_T < 1e-3 (ᾱ_T = 4.04e-05)
  ✓ Forward process: x_{T} ≈ N(0, I) (mean +0.004, std 1.000)
  ✓ U-Net output shape equals input shape ((2, 3, 32, 32))
  ✓ L_simple finite; every parameter gets a finite non-zero gradient (loss 0.9777)
  ✓ 10-step reverse pass finite and within [-1, 1]
  - Memory check skipped (--skip-memory)

Verification summary
├── Checks passed: 6/6
├── Parameters: 16,056,451 (16.06 M)
├── Measured GPU memory: n/a MiB (budget 3072)
└── Elapsed: 6.3 s (within the SC-001 target of 120 s)


In [ ]:
# 6. Train, resuming automatically from Drive if a checkpoint exists
import os, shlex
cmd = f"ddpm train --config configs/cifar10_colab.yaml --output-dir {shlex.quote(RUN_DIR)}"
latest = os.path.join(RUN_DIR, "latest.pt")
if os.path.exists(latest):
    cmd += f" --resume {shlex.quote(latest)}"
print(cmd)
!{cmd}

ddpm train --config configs/cifar10_colab.yaml --output-dir /content/drive/MyDrive/hands-on-ddpm/runs/cifar10_colab
100% 170M/170M [00:09<00:00, 18.7MB/s]
2026-10-03T15:39:37+0000 [INFO] ddpm.trainer./content/drive/MyDrive/hands-on-ddpm/runs/cifar10_colab: Training on cuda | epochs 1-100 | batch 128 × accum 1 | mixed precision none
2026-10-03T15:43:12+0000 [INFO] ddpm.trainer./content/drive/MyDrive/hands-on-ddpm/runs/cifar10_colab: Epoch 001 | train 0.9056 | val 0.7037 | val EMA 0.9979 | lr 1.41e-05 | 209s | 5690 MiB | best
2026-10-03T15:47:15+0000 [INFO] ddpm.trainer./content/drive/MyDrive/hands-on-ddpm/runs/cifar10_colab: Epoch 002 | train 0.4245 | val 0.1560 | val EMA 0.9850 | lr 2.81e-05 | 230s | 5794 MiB | best
2026-10-03T15:51:14+0000 [INFO] ddpm.trainer./content/drive/MyDrive/hands-on-ddpm/runs/cifar10_colab: Epoch 003 | train 0.0776 | val 0.0421 | val EMA 0.9568 | lr 4.22e-05 | 235s | 5794 MiB | best
2026-10-03T15:55:19+0000 [INFO] ddpm.trainer./content/drive/MyDrive/hands-on-d

In [ ]:
# 7. Optional: denoising strip and the 5,000-image FID/IS benchmark (~1-3 h depending on GPU)
#    Samples go to a folder named after the checkpoint's epoch; the benchmark refuses to reuse
#    batches whose manifest does not match the checkpoint, so stale images are never scored.
#    Raw weights: the EMA (0.9999) is not converged after ~80 epochs (see the report, section 4).
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    import torch
    best = os.path.join(RUN_DIR, "best_checkpoint.pt")
    epoch = torch.load(best, map_location="cpu", weights_only=False)["epoch"]
    samples = f"{EVAL_DIR}/samples_42_ep{epoch:03d}"
    !ddpm denoise-strip --checkpoint {best} --num-images 8 --seed 42 --weights raw --out {EVAL_DIR}/denoise_strip_ep{epoch:03d}.png
    !ddpm benchmark --checkpoint {best} --num-samples 5000 --weights raw --out {EVAL_DIR}/benchmark_metrics_ep{epoch:03d}.json --reuse-samples {samples}

In [ ]:
# 8. Show the loss curve and the most recent EMA sample grid
import glob, os
from IPython.display import Image, display
curve = os.path.join(RUN_DIR, "loss_curve.png")
if os.path.exists(curve):
    display(Image(curve))
grids = sorted(glob.glob(os.path.join(RUN_DIR, "samples", "epoch_*.png")))
if grids:
    print(grids[-1])
    display(Image(grids[-1]))